# Assignment 4: Insurance Purchase Prediction Web Application
**EMEA AI Cohort 12 — Hands-on Assignment**

## Objective
Train a Logistic Regression model to predict insurance purchase from customer attributes, save it as `model.pkl`, build a FastAPI backend and an HTML/CSS/JavaScript frontend, validate inputs, and demonstrate the application locally.

**Dataset:** UCI Bank Marketing dataset (UCI ID 222), using the `bank` variant. The target is whether the customer subscribed to a term deposit (`yes`/`no`), used here as a practical binary purchase/response prediction example.

> This is a proxy for the assignment's insurance purchase prediction scenario because the notebook does not have an insurance-specific dataset URL in the assignment image. Clearly disclose the dataset choice in your submission, or replace it with an approved insurance-purchase dataset if your trainer requires one. Metrics are calculated from the actual downloaded data and are not fabricated.

## 1. Install and import libraries

Run this notebook in Google Colab or Jupyter. Internet access is required to download the UCI dataset. The final app runs locally with FastAPI and Uvicorn.

In [ ]:
%pip install -q ucimlrepo pandas numpy scikit-learn joblib fastapi uvicorn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Download and prepare the dataset

The UCI Bank Marketing data has a mixture of numeric and categorical features. We remove `duration` because it is only known after a call and can create unrealistic prediction leakage if the goal is to predict before contact. The target is mapped to 1 for `yes` and 0 for `no`.

In [ ]:
repo = fetch_ucirepo(id=222)
X_raw = repo.data.features.copy()
y_raw = repo.data.targets.copy()

X = X_raw.copy()
X.columns = [str(c).strip().lower().replace(" ", "_") for c in X.columns]

# Avoid using call duration for a pre-contact prediction.
if "duration" in X.columns:
    X = X.drop(columns=["duration"])

target = y_raw.iloc[:, 0].astype(str).str.strip().str.lower()
y = target.map({"yes": 1, "no": 0})

valid = y.notna()
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype("int64").reset_index(drop=True)

# Normalize text and missing-value markers.
for col in X.select_dtypes(include=["object", "string"]).columns:
    X[col] = X[col].astype(object).apply(
        lambda value: np.nan if pd.isna(value) or str(value).strip() in {"", "?"}
        else str(value).strip()
    )

print("Dataset shape:", X.shape)
print("Target distribution (0 = no, 1 = yes):")
display(y.value_counts().rename_axis("subscribed").to_frame("count"))
display(X.head())

## 3. Split data and train the Logistic Regression model

Use a stratified 80/20 split. Numeric fields are median-imputed and scaled. Categorical fields are most-frequent-imputed and one-hot encoded. All transformations are fitted only on the training split.

In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Model training completed.")
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 4. Evaluate the model

Accuracy measures overall correctness. Precision, recall, and F1-score help assess the positive `yes` class, which may be less common than `no`. The confusion matrix shows the counts of correct and incorrect predictions.

In [ ]:
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (yes)", "Recall (yes)", "F1-score (yes)"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    ]
})
display(metrics.style.format({"Score": "{:.4f}"}))

print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["No subscription", "Yes subscription"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["No", "Yes"],
    cmap="Blues",
    values_format="d"
)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

## 5. Save the trained pipeline as `model.pkl`

Saving the entire pipeline ensures the backend applies the same imputation, scaling, and encoding steps as training. The app will load this file to predict from a form submission.

In [ ]:
model_path = Path("model.pkl")
joblib.dump(model, model_path)
print("Saved model to:", model_path.resolve())
print("File size (bytes):", model_path.stat().st_size)

## 6. Create the FastAPI backend and frontend files

Run the next cell to create a small project folder with:
- `app.py` — FastAPI backend and input validation
- `static/index.html` — form and result display
- `requirements.txt` — dependencies

The form fields are generated from the dataset's feature names. Numeric fields use number inputs; categorical fields use text inputs to keep the demo independent of hard-coded category lists. The API validates required fields and rejects malformed numeric inputs.

In [ ]:
from pathlib import Path
import json

project_dir = Path("insurance_prediction_app")
static_dir = project_dir / "static"
static_dir.mkdir(parents=True, exist_ok=True)

# Capture the training feature order and data types for backend validation.
feature_schema = {
    "numeric": numeric_features,
    "categorical": categorical_features,
    "all": X.columns.tolist()
}
(project_dir / "feature_schema.json").write_text(json.dumps(feature_schema, indent=2), encoding="utf-8")

app_code = r'''from pathlib import Path
import json
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR.parent / "model.pkl"
SCHEMA_PATH = BASE_DIR / "feature_schema.json"

model = joblib.load(MODEL_PATH)
schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
app = FastAPI(title="Insurance Purchase Prediction Demo")
app.mount("/static", StaticFiles(directory=BASE_DIR / "static"), name="static")

class PredictionInput(BaseModel):
    features: dict

@app.get("/", response_class=HTMLResponse)
def home():
    return (BASE_DIR / "static" / "index.html").read_text(encoding="utf-8")

@app.get("/schema")
def get_schema():
    return schema

@app.post("/predict")
def predict(payload: PredictionInput):
    missing = [col for col in schema["all"] if col not in payload.features]
    if missing:
        raise HTTPException(status_code=422, detail={"missing_fields": missing})

    row = {}
    for col in schema["all"]:
        value = payload.features[col]
        if col in schema["numeric"]:
            try:
                row[col] = float(value)
            except (TypeError, ValueError):
                raise HTTPException(status_code=422, detail=f"{col} must be numeric")
        else:
            if value is None or str(value).strip() == "":
                raise HTTPException(status_code=422, detail=f"{col} cannot be empty")
            row[col] = str(value).strip()

    frame = pd.DataFrame([row], columns=schema["all"])
    prediction = int(model.predict(frame)[0])
    probability = float(model.predict_proba(frame)[0][1])
    return {
        "prediction": "Likely to subscribe" if prediction == 1 else "Unlikely to subscribe",
        "class": prediction,
        "probability_yes": round(probability, 4),
        "note": "Demo trained on UCI Bank Marketing data, not an insurance-specific dataset."
    }
'''
(project_dir / "app.py").write_text(app_code, encoding="utf-8")

html_code = r'''<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Insurance Purchase Prediction Demo</title>
  <style>
    body { font-family: Arial, sans-serif; max-width: 900px; margin: 32px auto; padding: 0 16px; color: #202124; }
    h1 { margin-bottom: 8px; } .note { color: #5f6368; margin-bottom: 24px; }
    form { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 14px; }
    label { display: flex; flex-direction: column; gap: 6px; font-size: 14px; }
    input { padding: 10px; border: 1px solid #bbb; border-radius: 6px; }
    button { margin-top: 20px; padding: 12px 18px; border: 0; border-radius: 6px; cursor: pointer; background: #175cd3; color: white; }
    #result { margin-top: 20px; padding: 14px; border-radius: 6px; background: #f1f5f9; white-space: pre-wrap; }
    .error { color: #b42318; }
  </style>
</head>
<body>
  <h1>Insurance Purchase Prediction</h1>
  <p class="note">Educational demo using a UCI Bank Marketing model as a binary-response proxy. It is not an insurance underwriting tool.</p>
  <form id="prediction-form"></form>
  <button id="submit-button" type="submit" form="prediction-form">Predict</button>
  <div id="result" aria-live="polite">Enter the features and select Predict.</div>
  <script>
    const form = document.getElementById("prediction-form");
    const result = document.getElementById("result");
    let schema = null;

    async function initialize() {
      result.textContent = "Loading form fields...";
      try {
        const response = await fetch("/schema");
        if (!response.ok) throw new Error("Could not load input schema.");
        schema = await response.json();
        schema.all.forEach((name) => {
          const label = document.createElement("label");
          label.textContent = name.replaceAll("_", " ");
          const input = document.createElement("input");
          input.name = name;
          input.required = true;
          input.type = schema.numeric.includes(name) ? "number" : "text";
          if (input.type === "number") input.step = "any";
          label.appendChild(input);
          form.appendChild(label);
        });
        result.textContent = "Form ready. Complete all fields and select Predict.";
      } catch (error) {
        result.textContent = error.message;
        result.className = "error";
      }
    }

    form.addEventListener("submit", async (event) => {
      event.preventDefault();
      if (!schema) return;
      const features = {};
      for (const name of schema.all) {
        const input = form.elements.namedItem(name);
        features[name] = schema.numeric.includes(name) ? Number(input.value) : input.value.trim();
      }
      result.className = "";
      result.textContent = "Generating prediction...";
      try {
        const response = await fetch("/predict", {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify({ features })
        });
        const data = await response.json();
        if (!response.ok) throw new Error(JSON.stringify(data.detail ?? data));
        result.textContent =
          `Prediction: ${data.prediction}
Probability of yes: ${(data.probability_yes * 100).toFixed(2)}%

${data.note}`;
      } catch (error) {
        result.className = "error";
        result.textContent = `Prediction failed: ${error.message}`;
      }
    });
    initialize();
  </script>
</body>
</html>
'''
(static_dir / "index.html").write_text(html_code, encoding="utf-8")

(project_dir / "requirements.txt").write_text(
    "fastapi\nuvicorn\npandas\nscikit-learn\njoblib\npydantic\n",
    encoding="utf-8"
)

print("Created project files:")
for path in sorted(project_dir.rglob("*")):
    if path.is_file():
        print("-", path)

## 7. Prepare the model file for the app

The notebook saves `model.pkl` in the current working directory. The generated `app.py` expects `model.pkl` one level above the `insurance_prediction_app` folder, matching the notebook's default folder structure.

Your project structure should be:

```text
working_directory/
├── model.pkl
└── insurance_prediction_app/
    ├── app.py
    ├── feature_schema.json
    ├── requirements.txt
    └── static/
        └── index.html
```

For a local computer, download the `model.pkl` and the `insurance_prediction_app` folder (or recreate the files using the cell above).

## 8. Run the application locally

In a terminal, change directory to the folder containing `model.pkl` and `insurance_prediction_app`, then run:

```bash
python -m pip install -r insurance_prediction_app/requirements.txt
python -m uvicorn insurance_prediction_app.app:app --reload
```

Open `http://127.0.0.1:8000` in your browser. Use `http://127.0.0.1:8000/docs` to test the API interactively.

**Note:** Google Colab is useful for training and generating files, but `127.0.0.1` on your computer is not the same as the Colab runtime. For the required local demonstration, download the generated files and run the commands on your own computer. Do not expose the demo publicly without adding security controls.

## 9. Optional: create a ZIP package for download

Run this cell after the model and app files have been generated. It packages the trained model and application files into one ZIP file. The model file is generated by the actual training run.

In [ ]:
import zipfile

zip_path = Path("insurance_prediction_app_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("model.pkl", arcname="model.pkl")
    for path in project_dir.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path))

print("Created ZIP bundle:", zip_path.resolve())
print("Bundle size (bytes):", zip_path.stat().st_size)

## 10. Results and conclusion

### Results to record
After running the notebook, copy the measured accuracy, precision, recall, and F1-score from the evaluation table. Also record the confusion matrix counts and confirm that `model.pkl` and the application files were created.

### Conclusion template
> I developed a binary classification pipeline using Logistic Regression. The dataset was cleaned, split into training and testing sets, and preprocessed using median/mode imputation, scaling, and one-hot encoding. The model achieved **[insert measured accuracy]** accuracy, **[insert measured precision]** precision, **[insert measured recall]** recall, and **[insert measured F1-score]** F1-score on the held-out test set. I saved the complete preprocessing and model pipeline as `model.pkl` and created a FastAPI backend with an HTML/CSS/JavaScript frontend. The API validates input fields and returns a prediction with the estimated probability. The chosen UCI Bank Marketing dataset predicts term-deposit subscription rather than insurance purchase, so it is a functional demonstration of the required web-app architecture; an approved insurance-specific dataset should be substituted if required by the trainer.

### Submission checklist
- [ ] Run all notebook cells in order.
- [ ] Save the measured evaluation results and confusion matrix.
- [ ] Confirm `model.pkl` is created.
- [ ] Include `app.py`, `feature_schema.json`, `requirements.txt`, and `static/index.html`.
- [ ] Run the FastAPI app locally and test the form and `/docs`.
- [ ] Clearly disclose the dataset choice and do not claim it predicts insurance purchase unless an insurance dataset is used.